# ASF에서 Sentinel-1 GRD 다운로드

In [ ]:
# pip install asf_search shapely requests tqdm
import os, re, csv
import requests
import asf_search as asf
from shapely.geometry import Point, shape
from urllib.parse import urljoin
from tqdm.auto import tqdm
from asf_search.constants import INTERNAL

# ===== 타임아웃 여유 (CMR가 느릴 때 대비) =====
INTERNAL.CMR_TIMEOUT = 120  # 기본 30초 → 120초

# ===================== 사용자 설정 =====================
# ① 포인트(위도, 경도) — 이 점을 포함하는 씬만 선별
LAT, LON = 35.180386, 128.119025  # 위도, 경도(순서 주의: lat, lon)

# ② 시간 (UTC)
DATE_RANGE = "2025-07-12/2025-07-20"  # "YYYY-MM-DD/YYYY-MM-DD"

# ③ 제품/조건
PLATFORMS        = ["Sentinel-1A", "Sentinel-1B", "Sentinel-1C"]
BEAMMODE         = "IW"
FLIGHT_DIRECTION = None  # "ASCENDING" / "DESCENDING" / None
MAX_RESULTS      = 200

# ④ 출력
OUT_DIR  = r"<DATA_ROOT>\test\SEN1_GRD"
LIST_CSV = os.path.join(OUT_DIR, "sentinel1_point_GRD_candidates.csv")
DO_DOWNLOAD = True         # 목록만 보려면 False, 다운로드까지 하려면 True
CHUNK_MB    = 8            # 다운로드 청크 크기(MB)
os.makedirs(OUT_DIR, exist_ok=True)

# ⑤ Earthdata ID/PW (직접 입력)
EDL_USER = os.environ.get("EARTHDATA_USER", "")
EDL_PASS = os.environ.get("EARTHDATA_PASS", "")
# ======================================================


# ---------- 유틸 ----------
def detect_product_type(scene_name: str) -> str:
    m = re.search(r'_([A-Z]{3,4})(?:_|__)', scene_name)
    if not m:
        return ""
    tag = m.group(1)
    if tag.startswith("GRD"):
        return "GRD"
    return tag  # SLC/OCN/RAW

def datapool_folder_from_scene_GRD(scene_name: str):
    """
    GRD Datapool 폴더 규칙:
      상위: GRDH -> GRD_HD, GRDM -> GRD_MD
      하위(위성): S1A→SA, S1B→SB, S1C→SC
    """
    if "_GRDH_" in scene_name:
        top = "GRD_HD"
    elif "_GRDM_" in scene_name:
        top = "GRD_MD"
    else:
        top = None

    if   scene_name.startswith("S1A"): sat = "SA"
    elif scene_name.startswith("S1B"): sat = "SB"
    elif scene_name.startswith("S1C"): sat = "SC"
    else: sat = None
    return top, sat

def build_datapool_url_GRD(scene_name: str, file_name: str) -> str:
    top, sat = datapool_folder_from_scene_GRD(scene_name)
    if not top or not sat:
        raise ValueError(f"Datapool 경로를 유추할 수 없습니다: {scene_name}")
    base = f"https://datapool.asf.alaska.edu/{top}/{sat}/"
    return urljoin(base, file_name)

def fix_to_zip_name(name: str, scene: str) -> str:
    """
    ASF properties['fileName']가 .iso.xml로 오는 경우 → .zip으로 보정
    없거나 확장자 불분명하면 scene + '.zip'
    """
    if not name:
        return scene + ".zip"
    name2 = re.sub(r'\.iso\.xml$', '.zip', name, flags=re.IGNORECASE)
    if '.' not in os.path.basename(name2):
        name2 += '.zip'
    if not name2.lower().endswith('.zip'):
        name2 = os.path.splitext(name2)[0] + '.zip'
    return name2

def stream_download_with_progress(session: requests.Session, url: str, dst_path: str, chunk_mb: int = 8):
    """
    Datapool ZIP을 세션으로 스트리밍 다운로드하면서 tqdm로 진행률 표시.
    임시파일 .part로 저장 후 완료 시 원본명으로 rename.
    """
    tmp_path = dst_path + ".part"
    with session.get(url, stream=True) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("Content-Length", 0))
        chunk_size = chunk_mb * 1024 * 1024
        desc_name = os.path.basename(dst_path)
        with open(tmp_path, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B", unit_scale=True, unit_divisor=1024,
            desc=desc_name, leave=False
        ) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    os.replace(tmp_path, dst_path)


# ---------- 포인트 AOI ----------
wkt_point = f"POINT({LON} {LAT})"
start_date, end_date = DATE_RANGE.split("/")

# ---------- 검색 ----------
search_kwargs = dict(
    intersectsWith=wkt_point,     # 포인트와 교차(= 포함)하는 씬
    start=start_date,
    end=end_date,
    platform=PLATFORMS,
    beamMode=BEAMMODE,
    maxResults=MAX_RESULTS,
)
if FLIGHT_DIRECTION:
    search_kwargs["flightDirection"] = FLIGHT_DIRECTION

results = asf.geo_search(**search_kwargs)
print(f"\n검색된 씬 수(원본): {len(results)}")

# ---------- sceneName 중복 제거 ----------
unique = {}
for r in results:
    scene = r.properties.get("sceneName") or r.properties.get("fileName")
    if scene and scene not in unique:
        unique[scene] = r
results = list(unique.values())
print(f"중복 제거 후 씬 수: {len(results)}")

# ---------- 포인트 포함 여부(경계 포함: covers) + GRD만 선별 ----------
pt = Point(LON, LAT)
grd_items = []
for r in results:
    p = r.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    if detect_product_type(scene) != "GRD":
        continue
    geom_geojson = getattr(r, "geometry", None) or p.get("geometry")
    if not geom_geojson:
        continue
    scene_geom = shape(geom_geojson)
    if not scene_geom.covers(pt):  # 경계 위 점도 포함
        continue
    grd_items.append(r)

print(f"GRD 후보 수(포인트 포함): {len(grd_items)}")
if not grd_items:
    print("해당 포인트/시간 조건에 맞는 GRD 결과가 없습니다.")
    raise SystemExit()

# ---------- 목록 출력 + CSV (Path/Frame 포함) ----------
print("\n[다운로드 후보 목록 - 포인트 포함 | GRD | 편파 전체 | Path/Frame 포함]")
with open(LIST_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "scene", "utc", "polarization", "direction", "orbit",
        "relative_orbit", "path", "frame", "filename"
    ])
    for r in grd_items:
        p = r.properties
        scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
        utc   = (p.get("startTime") or "")[:19]
        pol   = p.get("polarization", "")
        dirn  = p.get("flightDirection", "")
        orbit = p.get("orbit", "")
        rel   = p.get("relativeOrbit", "NA")
        pathn = p.get("pathNumber", "NA")
        frame = p.get("frameNumber", "NA")
        fname = fix_to_zip_name(p.get("fileName"), scene)

        print(f"- {scene} | UTC:{utc} | 편파:{pol} | 방향:{dirn} | 궤도:{orbit} | "
              f"상대궤도:{rel} | Path:{pathn} | Frame:{frame} | 파일:{fname}")

        w.writerow([scene, utc, pol, dirn, orbit, rel, pathn, frame, fname])

print(f"\n후보 목록 CSV 저장: {LIST_CSV}")

# ---------- 여기서 멈춤(미리보기 모드) ----------
if not DO_DOWNLOAD:
    print("\n미리보기 전용 모드입니다. 다운로드를 진행하려면 DO_DOWNLOAD=True 로 바꾸세요.")
    raise SystemExit()

# ---------- Earthdata 인증 세션 (requests.Session 호환) ----------
session = asf.ASFSession().auth_with_creds(EDL_USER, EDL_PASS)

# ---------- ZIP 다운로드 (GRD_HD|GRD_MD/<SA|SB|SC>/..., tqdm 진행률 표시) ----------
downloaded_files = []
overall = tqdm(total=len(grd_items), desc="전체 진행", unit="file")

for product in grd_items:
    p = product.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    fname = fix_to_zip_name(p.get("fileName"), scene)  # iso.xml → zip 보정
    dst = os.path.join(OUT_DIR, fname)

    if os.path.exists(dst):
        tqdm.write(f"⏭ 이미 존재, 스킵: {dst}")
        overall.update(1)
        continue

    try:
        zip_url = build_datapool_url_GRD(scene, fname)
    except Exception as e:
        tqdm.write(f"⚠️ URL 생성 실패: {scene} → {e}")
        overall.update(1)
        continue

    try:
        tqdm.write(f"⬇ ZIP 다운로드: {scene}")
        stream_download_with_progress(session, zip_url, dst, chunk_mb=CHUNK_MB)
        if os.path.exists(dst):
            tqdm.write(f"✅ 완료: {dst}")
            downloaded_files.append(dst)
        else:
            tqdm.write(f"❌ 완료 표시 후 파일이 보이지 않습니다: {dst}")
    except Exception as e:
        tqdm.write(f"❌ 실패: {scene} → {e}")

    overall.update(1)

overall.close()

print("\n다운로드 완료 파일:")
for f in downloaded_files:
    print("  ", f)

print("\n완료.")


# ASF에서 Sentinel-1 SLC 다운로드

In [ ]:
# pip install asf_search shapely requests tqdm
import os, re, csv
import requests
import asf_search as asf
from shapely.geometry import Point, shape
from urllib.parse import urljoin, urlparse
from tqdm.auto import tqdm

# ===================== 사용자 설정 =====================
# ① 포인트(위도, 경도) — 이 점을 포함(>= 경계 포함)하는 씬만 선별
LAT, LON = 35.180386, 128.119025  # 위도, 경도(주의: lat, lon)

# ② 시간 (UTC)
DATE_RANGE = "2025-07-12/2025-07-20"  # "YYYY-MM-DD/YYYY-MM-DD"

# ③ 제품/조건
PLATFORMS        = ["Sentinel-1A", "Sentinel-1B", "Sentinel-1C"]
BEAMMODE         = "IW"
FLIGHT_DIRECTION = None  # "ASCENDING" / "DESCENDING" / None
MAX_RESULTS      = 200

# ④ 출력
OUT_DIR  = r"<DATA_ROOT>\test\SEN1_SLC"
LIST_CSV = os.path.join(OUT_DIR, "sentinel1_point_SLC_candidates.csv")
DO_DOWNLOAD = True   # 목록만 보려면 False, 다운로드까지 하려면 True
CHUNK_MB = 8         # 다운로드 청크 크기(MB)
os.makedirs(OUT_DIR, exist_ok=True)

# ⑤ Earthdata ID/PW (직접 입력)
EDL_USER = os.environ.get("EARTHDATA_USER", "")
EDL_PASS = os.environ.get("EARTHDATA_PASS", "")
# ======================================================


# ---------- 유틸들 ----------
def detect_product_type(scene_name: str) -> str:
    """
    S1A_IW_SLC__1SDV_... -> SLC
    S1A_IW_GRDH_1SDV_... -> GRD (참고)
    S1A_IW_OCN__2SDV_... -> OCN
    S1A_IW_RAW__0SDV_... -> RAW
    """
    m = re.search(r'_([A-Z]{3,4})(?:_|__)', scene_name)
    if not m:
        return ""
    tag = m.group(1)
    if tag.startswith("GRD"):
        return "GRD"
    return tag  # SLC/OCN/RAW

def datapool_folder_from_scene_SLC(scene_name: str):
    """
    SLC Datapool 폴더 규칙:
      상위: 'SLC'
      하위: S1A→SA, S1B→SB, S1C→SC
    """
    top = "SLC"
    if   scene_name.startswith("S1A"): sat = "SA"
    elif scene_name.startswith("S1B"): sat = "SB"
    elif scene_name.startswith("S1C"): sat = "SC"
    else: sat = None
    return top, sat

def build_datapool_url_SLC(scene_name: str, file_name: str) -> str:
    top, sat = datapool_folder_from_scene_SLC(scene_name)
    if not top or not sat:
        raise ValueError(f"Datapool 경로를 유추할 수 없습니다: {scene_name}")
    base = f"https://datapool.asf.alaska.edu/{top}/{sat}/"
    return urljoin(base, file_name)

def fix_to_zip_name(name: str, scene: str) -> str:
    """
    ASF properties['fileName']가 .iso.xml로 오는 경우 → .zip으로 보정
    없거나 확장자 불분명하면 scene + '.zip'
    """
    if not name:
        return scene + ".zip"
    name2 = re.sub(r'\.iso\.xml$', '.zip', name, flags=re.IGNORECASE)
    if '.' not in os.path.basename(name2):
        name2 += '.zip'
    if not name2.lower().endswith('.zip'):
        name2 = os.path.splitext(name2)[0] + '.zip'
    return name2

def stream_download_with_progress(session: requests.Session, url: str, dst_path: str, chunk_mb: int = 8):
    """
    Datapool ZIP을 세션으로 스트리밍 다운로드하면서 tqdm로 진행률 표시.
    임시파일 .part로 저장 후 완료 시 원본명으로 rename.
    """
    tmp_path = dst_path + ".part"
    # Range 재시작은 구현 안함(원하면 여기 추가 가능)
    with session.get(url, stream=True) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("Content-Length", 0))
        chunk_size = chunk_mb * 1024 * 1024
        desc_name = os.path.basename(dst_path)
        with open(tmp_path, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B", unit_scale=True, unit_divisor=1024,
            desc=desc_name, leave=False
        ) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    os.replace(tmp_path, dst_path)


# ---------- 포인트 AOI WKT ----------
wkt_point = f"POINT({LON} {LAT})"
start_date, end_date = DATE_RANGE.split("/")

# ---------- 검색 (편파 전체 포함) ----------
search_kwargs = dict(
    intersectsWith=wkt_point,     # 포인트와 교차(= 포함)하는 씬
    start=start_date,
    end=end_date,
    platform=PLATFORMS,
    beamMode=BEAMMODE,
    maxResults=MAX_RESULTS,
)
if FLIGHT_DIRECTION:
    search_kwargs["flightDirection"] = FLIGHT_DIRECTION

results = asf.geo_search(**search_kwargs)
print(f"\n검색된 씬 수(원본): {len(results)}")

# ---------- sceneName 기준 중복 제거 ----------
unique = {}
for r in results:
    scene = r.properties.get("sceneName") or r.properties.get("fileName")
    if scene and scene not in unique:
        unique[scene] = r
results = list(unique.values())
print(f"중복 제거 후 씬 수: {len(results)}")

# ---------- 포인트 포함 여부(엄격) + SLC만 선별 ----------
pt = Point(LON, LAT)
slc_items = []
for r in results:
    p = r.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    if detect_product_type(scene) != "SLC":
        continue
    geom_geojson = getattr(r, "geometry", None) or p.get("geometry")
    if not geom_geojson:
        continue
    scene_geom = shape(geom_geojson)
    # contains는 경계 위 점(False)을 제외하므로, 경계 포함 위해 covers 사용
    if not scene_geom.covers(pt):
        continue
    slc_items.append(r)

print(f"SLC 후보 수(포인트 포함): {len(slc_items)}")
if not slc_items:
    print("해당 포인트/시간 조건에 맞는 SLC 결과가 없습니다.")
    raise SystemExit()

# ---------- 목록 출력 + CSV( Path / Frame 포함 ) ----------
print("\n[다운로드 후보 목록 - 포인트 포함 | SLC | 편파 전체 | Path/Frame 포함]")
with open(LIST_CSV, "w", newline="", encoding="utf-8") as fcsv:
    w = csv.writer(fcsv)
    w.writerow([
        "scene", "utc", "polarization", "direction", "orbit",
        "relative_orbit", "path", "frame", "filename"
    ])
    for r in slc_items:
        p = r.properties
        scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
        utc   = (p.get("startTime") or "")[:19]
        pol   = p.get("polarization", "")
        dirn  = p.get("flightDirection", "")
        orbit = p.get("orbit", "")
        rel   = p.get("relativeOrbit", "NA")
        pathn = p.get("pathNumber", "NA")
        frame = p.get("frameNumber", "NA")
        fname = fix_to_zip_name(p.get("fileName"), scene)
        print(f"- {scene} | UTC:{utc} | 편파:{pol} | 방향:{dirn} | 궤도:{orbit} | Path:{pathn} | Frame:{frame} | 파일:{fname}")
        w.writerow([scene, utc, pol, dirn, orbit, rel, pathn, frame, fname])

print(f"\n후보 목록 CSV 저장: {LIST_CSV}")

# ---------- 여기서 멈춤(미리보기 모드) ----------
if not DO_DOWNLOAD:
    print("\n미리보기 전용 모드입니다. 다운로드를 진행하려면 DO_DOWNLOAD=True 로 바꾸세요.")
    raise SystemExit()

# ---------- Earthdata 인증 세션 ----------
session = asf.ASFSession().auth_with_creds(EDL_USER, EDL_PASS)

# ---------- ZIP 다운로드 (SLC/<SA|SB|SC>/..., tqdm 진행률 표시) ----------
downloaded_files = []
overall = tqdm(total=len(slc_items), desc="전체 진행", unit="file")

for product in slc_items:
    p = product.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    fname = fix_to_zip_name(p.get("fileName"), scene)  # iso.xml → zip 보정
    dst = os.path.join(OUT_DIR, fname)

    if os.path.exists(dst):
        tqdm.write(f"⏭ 이미 존재, 스킵: {dst}")
        overall.update(1)
        continue

    # Datapool URL 생성
    try:
        zip_url = build_datapool_url_SLC(scene, fname)
    except Exception as e:
        tqdm.write(f"⚠️ URL 생성 실패: {scene} → {e}")
        overall.update(1)
        continue

    try:
        tqdm.write(f"⬇ ZIP 다운로드: {scene}")
        stream_download_with_progress(session, zip_url, dst, chunk_mb=CHUNK_MB)
        if os.path.exists(dst):
            tqdm.write(f"✅ 완료: {dst}")
            downloaded_files.append(dst)
        else:
            tqdm.write(f"❌ 완료 표시 후 파일이 보이지 않습니다: {dst}")
    except Exception as e:
        tqdm.write(f"❌ 실패: {scene} → {e}")

    overall.update(1)

overall.close()

print("\n다운로드 완료 파일:")
for f in downloaded_files:
    print("  ", f)

print("\n완료.")


In [ ]:
# -*- coding: utf-8 -*-
# pip install asf_search shapely requests tqdm
import os, re, csv
import requests
import asf_search as asf
from shapely.geometry import Point, shape
from urllib.parse import urljoin
from tqdm.auto import tqdm
from asf_search.constants import INTERNAL

# =========================================================
# 설정 (여기만 바꿔 쓰세요)
# ---------------------------------------------------------
# ① AOI: 포인트(위도, 경도) — 이 점을 포함(경계 포함)하는 씬만 선별
LAT, LON = 35.180386, 128.119025   # (lat, lon)

# ② 시간 (UTC)
DATE_RANGE = "2025-07-12/2025-07-20"  # "YYYY-MM-DD/YYYY-MM-DD"

# ③ 제품/조건
PRODUCT_LEVEL   = "SLC"              # "GRD" 또는 "SLC"
PLATFORMS       = ["Sentinel-1A", "Sentinel-1B", "Sentinel-1C"]
BEAMMODE        = "IW"               # None 가능
FLIGHT_DIRECTION= None               # "ASCENDING" / "DESCENDING" / None
MAX_RESULTS     = 200

# ④ 출력
OUT_DIR   = r"<DATA_ROOT>\test\SEN1_MERGED"
CSV_NAME  = f"sentinel1_point_{PRODUCT_LEVEL}_candidates.csv"
DO_DOWNLOAD = True                   # 목록만 보려면 False
CHUNK_MB   = 8                       # 다운로드 청크 크기(MB)

# ⑤ Earthdata ID/PW (ASF Datapool용)
EDL_USER = os.environ.get("EARTHDATA_USER", "")
EDL_PASS = os.environ.get("EARTHDATA_PASS", "")
# =========================================================

# CMR 느릴 때 대비 타임아웃 증가
INTERNAL.CMR_TIMEOUT = 120

os.makedirs(OUT_DIR, exist_ok=True)
LIST_CSV = os.path.join(OUT_DIR, CSV_NAME)

# ---------------- 공통 유틸 ----------------
def detect_product_type(scene_name: str) -> str:
    """
    S1A_IW_SLC__1SDV_... -> SLC
    S1A_IW_GRDH_1SDV_... -> GRD
    """
    m = re.search(r'_([A-Z]{3,4})(?:_|__)', scene_name)
    if not m:
        return ""
    tag = m.group(1)
    return "GRD" if tag.startswith("GRD") else tag  # SLC/OCN/RAW 등

def datapool_folder_from_scene(scene_name: str, level: str):
    """
    ASF Datapool 폴더 규칙
      - SLC: 상위 'SLC' / 하위 S1A→SA, S1B→SB, S1C→SC
      - GRD: 상위 'GRD_HD' (GRDH) 또는 'GRD_MD' (GRDM) / 하위 S1A→SA, S1B→SB, S1C→SC
    """
    level = level.upper()
    if level == "SLC":
        top = "SLC"
    elif level == "GRD":
        if "_GRDH_" in scene_name:
            top = "GRD_HD"
        elif "_GRDM_" in scene_name:
            top = "GRD_MD"
        else:
            top = None
    else:
        top = None

    if   scene_name.startswith("S1A"): sat = "SA"
    elif scene_name.startswith("S1B"): sat = "SB"
    elif scene_name.startswith("S1C"): sat = "SC"
    else: sat = None

    return top, sat

def build_datapool_url(scene_name: str, file_name: str, level: str) -> str:
    top, sat = datapool_folder_from_scene(scene_name, level)
    if not top or not sat:
        raise ValueError(f"Datapool 경로를 유추할 수 없습니다: {scene_name} (level={level})")
    base = f"https://datapool.asf.alaska.edu/{top}/{sat}/"
    return urljoin(base, file_name)

def fix_to_zip_name(name: str | None, scene: str) -> str:
    """
    ASF properties['fileName']가 .iso.xml로 오는 경우 → .zip으로 보정.
    없거나 확장자 불분명하면 scene + '.zip'
    """
    if not name:
        return scene + ".zip"
    name2 = re.sub(r'\.iso\.xml$', '.zip', name, flags=re.IGNORECASE)
    if '.' not in os.path.basename(name2):
        name2 += '.zip'
    if not name2.lower().endswith('.zip'):
        name2 = os.path.splitext(name2)[0] + '.zip'
    return name2

def stream_download_with_progress(session: requests.Session, url: str, dst_path: str, chunk_mb: int = 8):
    """
    Datapool ZIP을 세션으로 스트리밍 다운로드 + tqdm 진행률
    """
    tmp_path = dst_path + ".part"
    with session.get(url, stream=True) as resp:
        resp.raise_for_status()
        total = int(resp.headers.get("Content-Length", 0))
        chunk_size = chunk_mb * 1024 * 1024
        desc_name = os.path.basename(dst_path)
        with open(tmp_path, "wb") as f, tqdm(
            total=total if total > 0 else None,
            unit="B", unit_scale=True, unit_divisor=1024,
            desc=desc_name, leave=False
        ) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))
    os.replace(tmp_path, dst_path)

# ---------------- 검색 준비 ----------------
wkt_point = f"POINT({LON} {LAT})"
start_date, end_date = DATE_RANGE.split("/")

search_kwargs = dict(
    intersectsWith=wkt_point,
    start=start_date,
    end=end_date,
    platform=PLATFORMS,
    beamMode=BEAMMODE,
    maxResults=MAX_RESULTS,
)
if FLIGHT_DIRECTION:
    search_kwargs["flightDirection"] = FLIGHT_DIRECTION

# ---------------- 검색 ----------------
results = asf.geo_search(**search_kwargs)
print(f"\n검색된 씬 수(원본): {len(results)}")

# sceneName 기준 중복 제거
unique = {}
for r in results:
    scene = r.properties.get("sceneName") or r.properties.get("fileName")
    if scene and scene not in unique:
        unique[scene] = r
results = list(unique.values())
print(f"중복 제거 후 씬 수: {len(results)}")

# 포인트 포함 여부(경계 포함: covers) + 원하는 레벨만 선별
pt = Point(LON, LAT)
picked = []
for r in results:
    p = r.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    if detect_product_type(scene) != PRODUCT_LEVEL.upper():
        continue
    geom_geojson = getattr(r, "geometry", None) or p.get("geometry")
    if not geom_geojson:
        continue
    scene_geom = shape(geom_geojson)
    if not scene_geom.covers(pt):   # 경계 위 점도 포함하려면 covers, 내부만이면 contains
        continue
    picked.append(r)

print(f"{PRODUCT_LEVEL} 후보 수(포인트 포함): {len(picked)}")
if not picked:
    print("해당 포인트/시간 조건에 맞는 결과가 없습니다.")
    raise SystemExit()

# ---------------- 목록 출력 + CSV (Path/Frame 포함) ----------------
print(f"\n[다운로드 후보 목록 - 포인트 포함 | {PRODUCT_LEVEL} | 편파 전체 | Path/Frame 포함]")
with open(LIST_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow([
        "scene", "utc", "polarization", "direction", "orbit",
        "relative_orbit", "path", "frame", "filename"
    ])
    for r in picked:
        p = r.properties
        scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
        utc   = (p.get("startTime") or "")[:19]
        pol   = p.get("polarization", "")
        dirn  = p.get("flightDirection", "")
        orbit = p.get("orbit", "")
        rel   = p.get("relativeOrbit", "NA")
        pathn = p.get("pathNumber", "NA")
        frame = p.get("frameNumber", "NA")
        fname = fix_to_zip_name(p.get("fileName"), scene)
        print(f"- {scene} | UTC:{utc} | 편파:{pol} | 방향:{dirn} | 궤도:{orbit} | 상대궤도:{rel} | "
              f"Path:{pathn} | Frame:{frame} | 파일:{fname}")
        w.writerow([scene, utc, pol, dirn, orbit, rel, pathn, frame, fname])

print(f"\n후보 목록 CSV 저장: {LIST_CSV}")

# ---------------- 다운로드 (옵션) ----------------
if not DO_DOWNLOAD:
    print("\n미리보기 전용 모드입니다. 다운로드를 진행하려면 DO_DOWNLOAD=True 로 바꾸세요.")
    raise SystemExit()

# Earthdata 인증 세션 (requests.Session 호환)
session = asf.ASFSession().auth_with_creds(EDL_USER, EDL_PASS)

downloaded_files = []
overall = tqdm(total=len(picked), desc="전체 진행", unit="file")
for product in picked:
    p = product.properties
    scene = p.get("sceneName") or p.get("fileName") or "UNKNOWN_SCENE"
    fname = fix_to_zip_name(p.get("fileName"), scene)
    dst = os.path.join(OUT_DIR, fname)

    if os.path.exists(dst):
        tqdm.write(f"⏭ 이미 존재, 스킵: {dst}")
        overall.update(1)
        continue

    try:
        zip_url = build_datapool_url(scene, fname, PRODUCT_LEVEL)
    except Exception as e:
        tqdm.write(f"⚠️ URL 생성 실패: {scene} → {e}")
        overall.update(1)
        continue

    try:
        tqdm.write(f"⬇ ZIP 다운로드: {scene}")
        stream_download_with_progress(session, zip_url, dst, chunk_mb=CHUNK_MB)
        if os.path.exists(dst):
            tqdm.write(f"✅ 완료: {dst}")
            downloaded_files.append(dst)
        else:
            tqdm.write(f"❌ 완료 표시 후 파일이 보이지 않습니다: {dst}")
    except Exception as e:
        tqdm.write(f"❌ 실패: {scene} → {e}")

    overall.update(1)

overall.close()

print("\n다운로드 완료 파일:")
for f in downloaded_files:
    print("  ", f)

print("\n완료.")
